In [ ]:
import os
import sys  
sys.path.append(os.path.abspath("..")) 
sys.path.append("<PROJECT_ROOT_D2>/")
from Funcs.Utility import *
import numpy as np
import pandas as pd
from typing import Dict, Callable, Union, Tuple, List, Optional, Iterable, Any
from datetime import timedelta as td
from scipy import stats
import ray
import warnings
import time

In [ ]:
def _safe_na_check(_v):
    _is_nan_inf = False
    
    try:
        _is_nan_inf = np.isnan(_v) or np.isinf(_v)
    except:
        _is_nan_inf = False
    
    return _is_nan_inf or _v is None

In [ ]:
DATA = load(os.path.join(PATH_INTERMEDIATE, 'proc_ESM.pkl'))

In [ ]:
DATA

Extraction functions

In [ ]:
def _extract_numeric_feature(d_key, d_val) -> Dict:
    feature = {}
    
    # Ensure the input is a NumPy array
    v = np.asarray(d_val)
    
    # Check if the data is numeric
    if not np.issubdtype(v.dtype, np.number):
        raise ValueError(f"Input data for {d_key} must be numeric.")
    
    # Handle NaNs and infinities
    if not np.all(np.isfinite(v)):
        raise ValueError(f"Input data for {d_key} contains NaNs or infinities.")
    
    # Calculate histogram
    hist, _ = np.histogram(v, bins='doane', density=False)
    
    # Calculate standard deviation
    std = np.sqrt(np.var(v, ddof=1)) if len(v) > 1 else 0
    
    # Normalize values
    v_norm = (v - np.mean(v)) / std if std != 0 else np.zeros(len(v))
    
    # Populate feature dictionary
    feature[f'{d_key}#AVG'] = np.mean(v) # Sample mean
    feature[f'{d_key}#STD'] = std # Sample standard deviation
    feature[f'{d_key}#SKW'] = stats.skew(v, bias=False) if std != 0 else 0 # Sample skewness
    feature[f'{d_key}#KUR'] = stats.kurtosis(v, bias=False) if std != 0 else 0 # Sample kurtosis
    feature[f'{d_key}#ASC'] = np.sum(np.abs(np.diff(v))) # Abstract sum of changes
    feature[f'{d_key}#BEP'] = stats.entropy(hist) # Binned entropy
    feature[f'{d_key}#MED'] = np.median(v) # Median
    feature[f'{d_key}#TSC'] = np.sqrt(np.sum(np.power(np.diff(v_norm), 2))) # Timeseries complexity
    
    return feature

In [ ]:
def _extract_categorical_feature(cats, d_key, d_val, is_bounded) -> Dict:
    feature = {}
    v = d_val
    cnt = v.value_counts()
    val, sup = cnt.index, cnt.values
    hist = {k: v for k, v in zip(val, sup)}

    # Information Entropy
    feature[f'{d_key}#ETP#'] = stats.entropy(sup / len(v))
    # Abs. Sum of Changes
    feature[f'{d_key}#ASC#'] = np.sum(v.values[1:] != v.values[:-1])
    if is_bounded:
        if len(cats) == 2: # Dichotomous categorical data
            c = cats[0]
            feature[f'{d_key}#RLV_SUP'] = hist[c] / len(v) if c in hist else 0
        else:
            for c in cats:
                feature[f'{d_key}#RLV_SUP={c}'] = hist[c] / len(v)  if c in hist else 0
            
    return feature

In [ ]:
def _extract_timeWindow_feature(is_numeric, cats, d_key, d_val) -> Dict:
    feature = {}
    v = d_val
    
    if d_key in ['SCR_EVENT']:
        # Extract features specifically for screen events
        s_on = v[v == 'SCREEN_ON'].index
        s_off = v[v == 'SCREEN_OFF'].index
        duration, onset, midpoint = calculate_sleep_duration(s_on, s_off, theta)
        
        if duration:
            feature['Sleep#Duration'] = duration
            onset_hour = onset.hour
            if onset_hour >= 21:
                feature['Sleep#Onset'] = onset_hour - 21
            else:
                feature['Sleep#Onset'] = onset_hour + 3
            feature['Sleep#Midpoint'] = midpoint.hour + midpoint.minute / 60
        else:
            feature['Sleep#Duration'] = 0
            feature['Sleep#Onset'] = 0
            feature['Sleep#Midpoint'] = 0
    else:
        if is_numeric:
            feature = _extract_numeric_feature(d_key, v)
        elif d_key in ['LOC_CLS']:
            is_bounded = False
            feature = _extract_categorical_feature(cats, d_key, v, is_bounded)
        else:
            is_bounded = True
            feature = _extract_categorical_feature(cats, d_key, v, is_bounded)

    return feature

In [ ]:
#This fucntion is based on the  towards circadian computing: "early to bed and early to rise"
#makes some of us unhealthy and sleep derived
theta=30
def calculate_sleep_duration(s_on, s_off, theta):
    # Merge s_on and s_off into a single DataFrame based on timestamp
    df = pd.merge(pd.DataFrame({'timestamp': s_on, 'event': 'SCREEN_ON'}),
                  pd.DataFrame({'timestamp': s_off, 'event': 'SCREEN_OFF'}),
                  how='outer', on='timestamp')
    # fill missing values in event_x with values from event_y, and vice versa
    df['event_x'] = df['event_x'].fillna(df['event_y'])
    df['event_y'] = df['event_y'].fillna(df['event_x'])
    # drop the event_x and event_y columns
    df = df.drop(columns=['event_y']).rename(columns={'event_x': 'event'})
    # Fill in missing timestamps with NaT and sort by timestamp
    df = df.fillna(pd.NaT).sort_values('timestamp')
    df=df.assign(
         timestamp=lambda x: pd.to_datetime(x['timestamp'], unit='ms', utc=True).dt.tz_convert(DEFAULT_TZ)
     )
    # Filter out screen-on events caused by notifications
    mask = (df['event'] == 'SCREEN_OFF') & ((df['timestamp'].diff().fillna(pd.NaT)  / pd.Timedelta(seconds=1)) > theta)
    filtered_df = df[mask].reset_index(drop=True)
    # Discard non-usage patterns that do not start between 9PM to 7AM (next day)
    sleep_duration = pd.Series(dtype=float)
    sleep_onset = pd.Series(dtype="datetime64[ns]")
    for i in range(len(filtered_df)-1):
        if filtered_df.loc[i, 'timestamp'].hour >= 21 or filtered_df.loc[i, 'timestamp'].hour < 7:
            non_usage_duration = filtered_df.loc[i+1, 'timestamp'] - filtered_df.loc[i, 'timestamp']
            if non_usage_duration.total_seconds() > 0:
                sleep_duration = pd.concat([sleep_duration, pd.Series(non_usage_duration.total_seconds())])
                sleep_onset = pd.concat([sleep_onset , pd.Series(filtered_df.loc[i, 'timestamp'])])
    # Calculate sleep midpoint and apply individual corrective term
    if len(sleep_duration) > 0:
        sleep_duration = sleep_duration.reset_index(drop=True)
        sleep_onset  =sleep_onset.reset_index(drop=True)
        sleep_midpoint = sleep_onset + pd.to_timedelta(sleep_duration/2, unit="s")
        return sleep_duration.max(), sleep_onset.iloc[sleep_duration.idxmax()], sleep_midpoint.iloc[sleep_duration.idxmax()]
    else:
        return None, None, None

In [ ]:
epoch_names = {
    0: 'Dawn',
    1: 'Morning',
    2: 'Afternoon',
    3: 'LateAfternoon',
    4: 'Evening',
    5: 'Night'
}
def _extract(
        pid: str,
        data: Dict[str, pd.Series],
        label: pd.Series,
        label_values: List[str],
#        window_data: Dict[str, Union[int, Callable[[pd.Timestamp], int]]],
#        window_label: Dict[str, Union[int, Callable[[pd.Timestamp], int]]],
        categories: Dict[str, Optional[List[any]]] = None,
        constant_features: Dict[str, any] = None,
        resample_s: Dict[str, float] = None
) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    _s = time.time()
    log(f"Begin feature extraction on {pid}'s data.")
    categories = categories or dict()
    constant_features = constant_features or dict()
    resample_s = resample_s or dict()
    X, y, date_times = [], [], []
#    count = 0
    for timestamp in label.index:
        row = dict()
        #Find the start of today and yesterday for extracting today epoch features and yesterday epoch features
        start_of_today = datetime(timestamp.year, timestamp.month, timestamp.day, tzinfo=timestamp.tzinfo)
        start_of_today = pd.Timestamp(start_of_today.date(), tz=DEFAULT_TZ)
        start_of_yesterday = timestamp - pd.Timedelta(days=1)
        start_of_yesterday = pd.Timestamp(start_of_yesterday.date(), tz=DEFAULT_TZ)
        label_cur = label.at[timestamp]
        t = timestamp - td(milliseconds=1)

        # #Yesterday and Today 3-hour epochs
        yesterday_time_windows_epoch = []
        for i in range(6):
            start = start_of_yesterday + pd.Timedelta(hours=i*3 + 6 )
            end = start_of_yesterday + pd.Timedelta(hours=(i+1)*3 +6)
            if start <= t:
                yesterday_time_windows_epoch.append((start, min(end, t)))
            else:
                break
        today_time_windows_epoch = []
        for i in range(6):
            start = start_of_today + pd.Timedelta(hours=i*3 +6)
            end = start_of_today + pd.Timedelta(hours=(i+1)*3 + 6)
            if start <= t:
                today_time_windows_epoch.append((start, min(end, t)))
            else:
                break
       # Yesterday and Today hourly windows
        yesterday_time_windows_hour = []
        for i in range(24):
            start = start_of_yesterday + pd.Timedelta(hours=i*1 )
            end = start_of_yesterday + pd.Timedelta(hours=(i+1)*1)
            if start <= t:
                yesterday_time_windows_hour.append((start, min(end, t)))
            else:
                break
        today_time_windows_hour = []
        for i in range(24):
            start = start_of_today + pd.Timedelta(hours=i*1 )
            end = start_of_today + pd.Timedelta(hours=(i+1)*1)
            if start <= t:
                today_time_windows_hour.append((start, min(end, t)))
            else:
                break
        
        # Features relevant to participants' info
        for d_key, d_val in constant_features.items():
            row[d_key] = d_val
            
        # Features from sensor data
        for d_key, d_val in data.items():
            is_numeric = d_key not in categories
            cats = categories.get(d_key) or list()
            d_val = d_val.sort_index()
            # Features relevant to latest value of a given data
            # These features are extracted only for bounded categorical data and numerical data.
            if is_numeric or cats:
                try:
                    v = d_val.loc[:t].iloc[-1]
                except (KeyError, IndexError):
                    v = 0
                if is_numeric:
                    row[f'{d_key}#VAL'] = v
                elif d_key not in ['LOC_CLS']:
                    for c in cats:
                        row[f'{d_key}#VAL={c}'] = v == c

            # Features relevant to duration since the latest state change.
            # These features are only for categorical data.
            # In addition, duration since a given state is set recently is considered,
            # that are available only at bounded categorical data.
            if not is_numeric:
                try:
                    v = d_val.loc[:t]
                    row[f'{d_key}#DSC'] = (t - v.index[-1]).total_seconds() if len(v) else -1.0
                    for c in cats:
                        # v_sub = v.loc[lambda x: x == c].index
                        if isinstance(v, pd.DataFrame):
                                mask = v.eq(c).any(axis=1)      # True if *any* column equals c
                        else:                               # v is a Series
                                mask = v == c
                        v_sub = v.index[mask]
                        row[f'{d_key}#DSC={c}'] = (t - v_sub[-1]).total_seconds() if len(v_sub) else -1.0
                except (KeyError, IndexError):
                    row[f'{d_key}#DSC'] = 0
                    for c in cats:
                        row[f'{d_key}#DSC={c}'] = 0
            
            # # Make d_val’s index match timezone
            # d_val.index = pd.to_datetime(d_val.index)
            # d_val.index = d_val.index.tz_convert(DEFAULT_TZ)

            #Features extracted from time-windows
            #These features requires resampling and imputation on each data.
            # sample_rate = resample_s.get(d_key) or 1
            # d_val_res = d_val.resample(f'{sample_rate}S', origin='start')
            
            
            #Features extracted from time-windows
            #These features requires resampling and imputation on each data.
            sample_rate = resample_s.get(d_key, '1s')

            # Handle duplicate timestamps before resampling
            if d_val.index.duplicated().any():
                # Keep the last occurrence of each duplicate timestamp
                d_val = d_val[~d_val.index.duplicated(keep='last')]

            # d_val_res = d_val.resample(f'{sample_rate}s', origin='start')  # Changed 'S' to 's' to fix deprecation warning
            d_val_res = d_val.resample(sample_rate, origin='start')
            
            if is_numeric:
                try:
                    d_val_res = d_val_res.mean().interpolate(method='linear').dropna()
                except ValueError:
                    # Save input data to a file or external storage for debugging...
                    print(d_val_res)
                    print(d_val)
                    raise
            else:
                d_val_res = d_val_res.ffill().dropna()
            #No resampling
            d_val_res =d_val

            ###############################################################    
    #            # Features extracted from 5-min immediate past time-windows
    #             w_val = 5 * 60
    #             try:
    #                 v = d_val_res.loc[t - td(seconds=w_val):t]
    #             except (KeyError, IndexError):
    #                 continue
    #             with warnings.catch_warnings():
    #                 warnings.simplefilter('ignore')
    #                 new_row = {f'{k}#ImmediatePast_5': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
    #                 row.update(new_row)
    #             # Features extracted from 10-min immediate past time-windows
    #             w_val = 10 * 60
    #             try:
    #                 v = d_val_res.loc[t - td(seconds=w_val):t]
    #             except (KeyError, IndexError):
    #                 continue
    #             with warnings.catch_warnings():
    #                 warnings.simplefilter('ignore')
    #                 new_row = {f'{k}#ImmediatePast_10': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
    #                 row.update(new_row)

            # Features extracted from 15-min immediate past time-windows
            w_val = 15 * 60
            try:
                v = d_val_res.loc[t - td(seconds=w_val):t]
            except (KeyError, IndexError):
                continue
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                new_row = {f'{k}#ImmediatePast_15': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
                row.update(new_row)

                # Features extracted from 30-min immediate past time-windows
                w_val = 30 * 60
                try:
                    v = d_val_res.loc[t - td(seconds=w_val):t]
                except (KeyError, IndexError):
                    continue
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    new_row = {f'{k}#ImmediatePast_30': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
                    row.update(new_row)

    #             # Features extracted from 45-min immediate past time-windows
    #             w_val = 45 * 60
    #             try:
    #                 v = d_val_res.loc[t - td(seconds=w_val):t]
    #             except (KeyError, IndexError):
    #                 continue
    #             with warnings.catch_warnings():
    #                 warnings.simplefilter('ignore')
    #                 new_row = {f'{k}#ImmediatePast_45': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
    #                 row.update(new_row)

            #############################################################    
            #Features extracted from yesterday epoch time windows
            for count, (start, end) in enumerate(yesterday_time_windows_epoch):
                # Get data for the current yesterday epoch time window
                try:
                    v = d_val_res.loc[start:end]
                except (KeyError, IndexError):
                    continue
                epoch_name = epoch_names.get(count)

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    new_row = {f'{k}#Yesterday{epoch_name}': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
                    row.update(new_row)
                
            #Features extracted from today epoch time windows until current time
            for count, (start, end) in enumerate(today_time_windows_epoch):
                # Get data for the current time window
                try:
                    v = d_val_res.loc[start:end]
                except (KeyError, IndexError):
                    continue
                epoch_name = epoch_names.get(count)

                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    new_row = {f'{k}_Today{epoch_name}': v for k, v in _extract_timeWindow_feature(is_numeric, cats, d_key, v).items()}
                    row.update(new_row)
            #Sleep feature extracted from last night's data
            onset_min = start_of_yesterday + pd.Timedelta(hours=21)
            onset_max = start_of_today + pd.Timedelta(hours=14)
            s_on =data['SCR_EVENT'].loc[data['SCR_EVENT']=='ON']
            s_off =data['SCR_EVENT'].loc[data['SCR_EVENT']=='OFF']
            duration, onset, midpoint =calculate_sleep_duration(s_on.loc[onset_min:onset_max].reset_index()['timestamp'], s_off.loc[onset_min:onset_max].reset_index()['timestamp'], theta)
            if duration:
                row['Sleep#Duration'] = duration
                onset_hour = onset.hour
                if onset_hour >=21:
                    row['Sleep#Onset'] = onset_hour - 21
                else:
                    row['Sleep#Onset'] = onset_hour + 3
            else:
                row['Sleep#Duration'] = 0
                row['Sleep#Onset'] = 0
                
            # Features relevant to time
            day_of_week = ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN'][t.isoweekday() - 1]
            is_weekend = 'Y' if t.isoweekday() > 5 else 'N'
            hour = t.hour

            if 6 <= hour < 9:
                hour_name = 'Dawn'
            elif 9 <= hour < 12:
                hour_name = 'MORNING'
            elif 12 <= hour < 15:
                hour_name = 'AFTERNOON'
            elif 15 <= hour < 18:
                hour_name = 'LATE_AFTERNOON'
            elif 18 <= hour < 21:
                hour_name = 'EVENING'
            elif 21 <= hour < 24:
                hour_name = 'NIGHT'
            else:
                hour_name = 'MIDNIGHT'
                
            for d in ['MON', 'TUE', 'WED', 'THU', 'FRI', 'SAT', 'SUN']:
                row[f'Time#DOW={d}'] = d == day_of_week
            for d in ['Y', 'N']:
                row[f'Time#WKD={d}'] = d == is_weekend
            for d in ['DAWN', 'MORNING', 'AFTERNOON', 'LATE_AFTERNOON', 'EVENING', 'NIGHT', 'MIDNIGHT']:
                row[f'Time#HRN={d}'] = d == hour_name
            

            # try:
            #     last_label = label.loc[label[:t].index.max()]
            # except (KeyError, IndexError):
            #     last_label = 0
            # row[f'ESM#LastLabel'] = last_label

    # #############################################################################################
    #         #The following code is designed for fixed threshold
    #         # Label values extracted from yesterday epochs
    #         for count, (start, end) in enumerate(yesterday_time_windows_epoch):
    #             try:
    #                 v = label.loc[start:end]
    #                 epoch_name = epoch_names.get(count)
    #                 if len(label_values) <= 2: # Binary classification
    #                     row[f'ESM#LIK#Yesterday{epoch_name}'] = np.sum(v == label_values[0]) / len(v) if len(v) > 0 else 0
    #                 else:
    #                     for l in label_values:
    #                         row[f'ESM#LIK#Yesterday{epoch_name}'] = np.sum(v == l) / len(v) if len(v) > 0 else 0
    #             except (KeyError, IndexError):
    #                 epoch_name = epoch_names.get(count)
    #                 if len(label_values) <= 2:
    #                     row[f'ESM#LIK#Yesterday{epoch_name}'] = 0
    #                 else:
    #                     for l in label_values:
    #                         row[f'ESM#LIK#Yesterday{epoch_name}'] = 0
    #         # Label values extracted from today epochs
    #         for count, (start, end) in enumerate(today_time_windows_epoch):
    #             try:
    #                 v = label.loc[start:end]
    #                 epoch_name = epoch_names.get(count)
    #                 if len(label_values) <= 2: # Binary classification
    #                     row[f'ESM#LIK#Today{epoch_name}'] = np.sum(v == label_values[0]) / len(v) if len(v) > 0 else 0
    #                 else:
    #                     for l in label_values:
    #                         row[f'ESM#LIK#Today{epoch_name}'] = np.sum(v == l) / len(v) if len(v) > 0 else 0
    #             except (KeyError, IndexError):
    #                 epoch_name = epoch_names.get(count)
    #                 if len(label_values) <= 2:
    #                     row[f'ESM#LIK#Today{epoch_name}'] = 0
    #                 else:
    #                     for l in label_values:
    #                         row[f'ESM#LIK#Today{epoch_name}'] = 0
    # ###############################################################################3
            #The following code is designed for stress_dyn since we should not know previous binned label values

            # Label values extracted from yesterday epochs
    #         for count, (start, end) in enumerate(yesterday_time_windows_epoch):
    #             try:
    #                 v = label.loc[start:end]
    #                 epoch_name = epoch_names.get(count)
    #                 row[f'ESM#Mean#Yesterday{epoch_name}'] = np.mean(v) if len(v) > 0 else 0
    #                 row[f'ESM#Std#Yesterday{epoch_name}'] = np.std(v) if len(v) > 0 else 0

    #             except (KeyError, IndexError):
    #                 epoch_name = epoch_names.get(count)
    #                 row[f'ESM#Mean#Yesterday{epoch_name}'] = 0 
    #                 row[f'ESM#Std#Yesterday{epoch_name}'] = 0
                    
    # #         # Label values extracted from today epochs
    #         for count, (start, end) in enumerate(today_time_windows_epoch):
    #             try:
    #                 v = label.loc[start:end]
    #                 epoch_name = epoch_names.get(count)
    #                 row[f'ESM#Mean#Today{epoch_name}'] = np.mean(v) if len(v) > 0 else 0
    #                 row[f'ESM#Std#Today{epoch_name}'] = np.std(v) if len(v) > 0 else 0

    #             except (KeyError, IndexError):
    #                 epoch_name = epoch_names.get(count)
    #                 row[f'ESM#Mean#Today{epoch_name}'] = 0  
    #                 row[f'ESM#Std#Today{epoch_name}'] = 0 
    ############################################################################

    #         # Label values extracted from immediate past
    #         w_val = 15 * 60
    #         try:
    #             v = label.loc[t - td(seconds=w_val):t]
    #             epoch_name = epoch_names.get(count)
    #             if len(label_values) <= 2: # Binary classification
    #                 row[f'ESM#LIK#ImmediatePast'] = np.sum(v == label_values[0]) / len(v) if len(v) > 0 else 0
    #             else:
    #                 for l in label_values:
    #                     row[f'ESM#LIK={l}#ImmediatePast'] = np.sum(v == l) / len(v) if len(v) > 0 else 0
    #         except (KeyError, IndexError):
    #             epoch_name = epoch_names.get(count)
    #             if len(label_values) <= 2:
    #                 row[f'ESM#LIK#ImmediatePast'] = 0
    #             else:
    #                 for l in label_values:
    #                     row[f'ESM#LIK={l}#ImmediatePast'] = 0

        row = {
            k: 0 if _safe_na_check(v) else v
            for k, v in row.items()
        }

        X.append(row)
        y.append(label_cur)
        date_times.append(timestamp)
    
    log(f"Complete feature extraction on {pid}'s data ({time.time() - _s:.2f} s).")
    
    #Without normalization for each user
    X = pd.DataFrame(X)
    y = np.asarray(y)
    group = np.repeat(pid, len(y))
    date_times =  np.asarray(date_times)

#     #Normalization for each feature of user pid
#     df_X = pd.DataFrame(X).fillna(0)
#     df_X_sensor =  df_X.loc[:,[('PIF' not in str(x)) and ('ESM' not in str(x)) for x in df_X.keys()]]  
#     df_X_no_sensor = df_X.loc[:,[('PIF' in str(x)) or ('ESM' in str(x)) for x in df_X.keys()]]
#     # Z-score Standardization for each column
#     X_normalized_sensor = df_X_sensor.apply(lambda col: (col - col.mean()) / col.std())
#     X_normalized = pd.concat([X_normalized_sensor, df_X_no_sensor], axis=1 )
#     X, y, group, date_times = X_normalized, np.asarray(y), np.repeat(pid, len(y)), np.asarray(date_times)
    return X, y, group, date_times

In [ ]:
def extract(
        pids: Iterable[str],
        data: Dict[str, pd.Series],
        label: pd.Series,
        label_values: List[str],
#        window_data: Dict[str, Union[int, Callable[[pd.Timestamp], int]]],
#        window_label: Dict[str, Union[int, Callable[[pd.Timestamp], int]]],
        categories: Dict[str, Optional[List[any]]] = None,
        constat_features: Dict[str, Dict[str, any]] = None,
        resample_s: Dict[str, float] = None,
        with_ray: bool=False
):
    if with_ray and not ray.is_initialized():
        raise EnvironmentError('Ray should be initialized if "with_ray" is set as True.')
    func = ray.remote(_extract).remote if with_ray else _extract
    jobs = []
    for pid in pids:
        d = dict()
        for k, v in data.items():
            try:
                d[k] = v.loc[(pid, )]
                if k.startswith('LOC_'):
                    d[k].index= pd.to_datetime( d[k].index, unit='ms', utc=True).tz_convert(DEFAULT_TZ)
                d['SPEED'] = d.pop('LOC_SPEED')
            except (KeyError, IndexError):
                pass


        job = func(
            pid=pid, data=d, label=label.loc[(pid, )],
            label_values=label_values,
#            window_data=window_data,
#            window_label=window_label,
            categories=categories,
            constant_features=constat_features[pid],
            resample_s=resample_s
        )
        jobs.append(job)
    jobs = ray.get(jobs) if with_ray else jobs
    print([x.shape for _, x, _, _ in jobs])
    X = pd.concat([x for x, _, _, _ in jobs], axis=0, ignore_index=True)
    y = np.concatenate([x for _, x, _, _ in jobs], axis=0)
    group = np.concatenate([x for _, _, x, _ in jobs], axis=0)
    date_times = np.concatenate([x for _, _, _, x in jobs], axis=0)
    t_s = date_times.min().normalize().timestamp()
    t_norm = np.asarray(list(map(lambda x: x.timestamp() - t_s, date_times)))
    C, DTYPE = X.columns, X.dtypes
    X = X.fillna({
        **{c: False for c in C[(DTYPE == object) | (DTYPE == bool)]},
        **{c: 0.0 for c in C[(DTYPE != object) & (DTYPE != bool)]},
    }).astype({
        **{c: 'bool' for c in C[(DTYPE == object) | (DTYPE == bool)]},
        **{c: 'float32' for c in C[(DTYPE != object) & (DTYPE != bool)]},
    })
    return X, y, group, t_norm, date_times

In [ ]:
import os
import cloudpickle
import pandas as pd

LABEL_VALUES = [1, 0]

RESAMPLE_S = {
    'CAL': '10S',  # 1 second (was 1.0)
    'APP_DUR_UNKNOWN': '10S',  # 1 second (was 1.0)
    'BAT_LEV': '10S',  # 1 second (was 1.0)
    'MSG_RCV': '1T',  # 1 minute (was 60.0)
    'DATA_RCV': '10S',  # 10 seconds (was 10.0)
    'HEARTRATE': '10S',  # 1 second (was 1.0)
}


In [ ]:

# Define the path to the user info file
user_info_file = os.path.join(PATH_PARTICIPANT)

# Load the user info data
userinfo = pd.read_csv(user_info_file)

# Process participant information
PINFO = userinfo.set_index('pcode').assign(
    AGE=lambda x: x['age'],
    GEN=lambda x: x['gender'],
    BFI_OPN=lambda x: x['openness'],
    BFI_CON=lambda x: x['conscientiousness'],
    BFI_NEU=lambda x: x['neuroticism'],
    BFI_EXT=lambda x: x['extraversion'],
    BFI_AGR=lambda x: x['agreeableness'],
    GHQ=lambda x: x['GHQ12'],
    PSS=lambda x: x['PSS10'],
    PHQ=lambda x: x['PHQ9'],
    SWLS_PRESENT=lambda x: x['SWLS-present'],
    SWLS_PAST=lambda x: x['SWLS-past'],
    SWLS_OVERALL=lambda x: x['SWLS-overall'],
    RSES=lambda x: x['RSES'],
    SE=lambda x: x['PPC-self_efficacy'],
    OPT=lambda x: x['PPC-optimism'],
    HOPE=lambda x: x['PPC-hope'],
    RES=lambda x: x['PPC-resiliency']
)

# Convert the processed info into a dictionary
PINFO = pd.get_dummies(PINFO, prefix_sep='=', dtype=bool).to_dict('index')
PINFO = {k: {f'PIF#{x}': y for x, y in v.items()} for k, v in PINFO.items()}
DATA = load(os.path.join(PATH_INTERMEDIATE, 'proc_ESM.pkl'))
# LABELS_PROC = pd.read_csv(os.path.join(PATH_INTERMEDIATE, 'labels_1h_esmsyn.csv'), index_col=['pcode','timestamp'],parse_dates=True)
LABELS_PROC = pd.read_csv(os.path.join(PATH_INTERMEDIATE, 'labels_1h_esmsyn.csv'), parse_dates=True)

In [ ]:

def standardize_timestamp_label(df):
    """
    Converts the 'window_end_time' column in df to an exactly standardized datetime.
    It forces every timestamp to be in UTC, then removes the timezone info so that
    all dataframes have consistent, naïve datetime values representing the same moment in time.
    """
    # Use utc=True to force conversion into UTC timezone.
    df['timestamp'] = pd.to_datetime(
        df['timestamp'], 
        errors='raise',            # Raise an exception if conversion fails.
        format='mixed',
        utc=True,                  
        infer_datetime_format=True
    ).dt.tz_convert( DEFAULT_TZ)
        

    # Remove timezone (i.e. make timezone-naïve) while preserving the UTC timestamp.
    # df['timestamp'] = df['timestamp'].dt.tz_localize(None)
    
    return df


In [ ]:
LABELS_PROC = standardize_timestamp_label(LABELS_PROC)
LABELS_PROC = LABELS_PROC.set_index(['pcode', 'timestamp'], inplace=False).sort_index()

In [ ]:
def standardize_timestamp_data(dict_series):
    """
    Ensure the 'timestamp' level of each MultiIndex Series in dict_series is:
      1. Parsed to datetime.
      2. Converted to the target timezone (default UTC).

    Parameters
    ----------
    dict_series : dict[str, pd.Series]
        Mapping of keys (e.g., participant IDs) -> Series with MultiIndex.

    Returns
    -------
    dict[str, pd.Series]
        The updated dictionary with each Series standardized.
    """
    out = {}

    for key, series in dict_series.items():
        if 'timestamp' not in series.index.names:
            out[key] = series
            continue

        # Reset index for safer timestamp operations
        idx_names = list(series.index.names)
        tmp = series.reset_index()

        # Parse and standardize timestamps
        tmp['timestamp'] = pd.to_datetime(
            tmp['timestamp'],
            errors='raise',
            format='mixed',
            utc=True,
            infer_datetime_format=True
        ).dt.tz_convert(DEFAULT_TZ)

        # Restore original MultiIndex
        tmp = tmp.set_index(idx_names).sort_index()

        out[key] = tmp.iloc[:, 0]

    return out

In [ ]:
DATA = standardize_timestamp_data(DATA)

In [ ]:
import pandas as pd

def auto_extract_categories(data_dict):
    categories = {}
    for sensor_type, series in data_dict.items():
        if pd.api.types.is_numeric_dtype(series):
            continue  # Skip numeric data
        elif pd.api.types.is_object_dtype(series) or pd.api.types.is_categorical_dtype(series):
            unique_vals = series.dropna().unique().tolist()
            categories[sensor_type] = unique_vals
    return categories

# Example Usage:
CATEGORIES = auto_extract_categories(DATA)


In [ ]:
import warnings
from pandas.errors import PerformanceWarning

warnings.simplefilter(action='ignore', category=PerformanceWarning)
warnings.simplefilter(action="ignore", category=RuntimeWarning)

from Funcs.Utility import PATH_INTERMEDIATE, on_ray, dump
sys.path.append("<PROJECT_ROOT_D2>/")   # keep this for the driver

with on_ray(runtime_env={
        "working_dir": "<PROJECT_ROOT_D2>/",   # zip + add to PYTHONPATH
        "excludes": [
            "Intermediate/proc_hourly.pkl",
            "Intermediate/hourly_data_all/*.csv",
            "Intermediate/hourly_data_all/",
            "*.pkl",
            "*.csv",
            "__pycache__/",
            ".ipynb_checkpoints/",
            "*.log"
        ]
    }):
# with on_ray():
    # for l in ['stress_binary_personal', 'step_count_binary_personal']:
    for l in ['stress_binary_personal']:
        #In preprocessing, dynamic threshold shows better data balance
        labels = LABELS_PROC[f'{l}']
#         labels = LABELS_PROC['stress_fixed']
        pids = labels.index.get_level_values('pcode').unique()
        feat = extract(
            pids=pids,
            data=DATA,
            label=labels,
            label_values=LABEL_VALUES,
#            window_data=WINDOW_DATA,
#            window_label=WINDOW_LABEL,
            categories=CATEGORIES,
            constat_features=PINFO,
            resample_s=RESAMPLE_S,
            with_ray=True
        )
        # dump(feat, os.path.join(PATH_INTERMEDIATE, f'{l}-current.pkl'))
        dump(feat, os.path.join(PATH_INTERMEDIATE, f'{l}-full.pkl'))

In [ ]:
X, y, groups, t, date_times = load(os.path.join(PATH_INTERMEDIATE, f'stress_binary_personal-full.pkl'))
X_current = X[[key for key in X.columns if any(sub in key for sub in ['PIF', 'Sleep', '#VAL'])]]
feat = X_current, y, groups, t, date_times
dump(feat, os.path.join(PATH_INTERMEDIATE, 'stress_binary_personal-current.pkl'))